# 02 - Identification and Instance Segmentation
## Notebook Objective
This notebook implements the supervised learning phase of the pipeline. Using the patches and binary masks generated in Step 1, we will fine-tune a **Mask R-CNN** (with a ResNet50-FPN backbone). 

As discussed in the course, Mask R-CNN is an extension of Faster R-CNN that performs:
1. **Object Detection (Identification):** Predicting bounding boxes via the Region Proposal Network (RPN).
2. **Instance Segmentation:** Predicting pixel-exact masks for each detected bounding box using a Fully Convolutional branch and ROI Align.

We leverage Transfer Learning by starting with COCO pre-trained weights and adapting the classification heads to our specific binary task: Background (Class 0) vs Glomerulus (Class 1).

In [ ]:
import os
import cv2
import numpy as np
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import torchvision.transforms.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Check hardware acceleration (CUDA for cluster, MPS for Apple Silicon, CPU fallback)
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("Using GPU: CUDA")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using GPU: Apple MPS")
else:
    device = torch.device('cpu')
    print("Using CPU")

### 1. Custom Dataset Class
Mask R-CNN in PyTorch requires a very specific target dictionary during training. For each image, we must provide:
* `boxes`: Bounding box coordinates `[xmin, ymin, xmax, ymax]`.
* `masks`: A 3D tensor of binary masks (one for each instance of a glomerulus in the patch).
* `labels`: The class of each instance (always 1 for Glomerulus).

In [ ]:
import os
import cv2
import numpy as np
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image
import torchvision.transforms.functional as F

# Definiamo le cartelle (create dallo step 01)
images_dir = "./dataset_glomeruli_all/images"
masks_dir = "./dataset_glomeruli_all/masks"

# Otteniamo tutti i percorsi
all_imgs = sorted([os.path.join(images_dir, f) for f in os.listdir(images_dir) if f.endswith('.png')])
all_masks = sorted([os.path.join(masks_dir, f) for f in os.listdir(masks_dir) if f.endswith('.png')])

# Usiamo train_test_split di sklearn (80% train, 20% val)
train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    all_imgs, all_masks, test_size=0.2, random_state=42
)

print(f"Total images: {len(all_imgs)}")
print(f"Training set: {len(train_imgs)}")
print(f"Validation set: {len(val_imgs)}")

In [ ]:
class GlomeruliDataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        # Invece delle cartelle, il dataset riceve direttamente le liste splittate
        self.image_paths = image_paths
        self.mask_paths = mask_paths

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]
        
        img = Image.open(img_path).convert("RGB")
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        # Binarizzazione sicura
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

        # Gestione di multipli glomeruli (Connected Components)
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
        
        obj_ids = list(range(1, num_labels)) # Salta lo zero (sfondo)
        num_objs = len(obj_ids)

        # --- FIX: GESTIONE DEI NEGATIVE SAMPLES (SFONDO VUOTO) ---
        if num_objs == 0:
            # Creiamo tensori vuoti ma con le DIMENSIONI CORRETTE richieste da Mask R-CNN
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            masks = torch.zeros((0, mask.shape[0], mask.shape[1]), dtype=torch.uint8)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = []
            masks = np.zeros((num_objs, mask.shape[0], mask.shape[1]), dtype=np.uint8)

            for i, obj_id in enumerate(obj_ids):
                obj_mask = (labels == obj_id).astype(np.uint8)
                masks[i] = obj_mask
                
                x = stats[obj_id, cv2.CC_STAT_LEFT]
                y = stats[obj_id, cv2.CC_STAT_TOP]
                w = stats[obj_id, cv2.CC_STAT_WIDTH]
                h = stats[obj_id, cv2.CC_STAT_HEIGHT]
                boxes.append([x, y, x + w, y + h])

            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.ones((num_objs,), dtype=torch.int64) # Class 1: Glomerulus
            masks = torch.as_tensor(masks, dtype=torch.uint8)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros((num_objs,), dtype=torch.int64)

        image_id = torch.tensor([idx])

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }

        img = F.to_tensor(img)

        return img, target

    def __len__(self):
        return len(self.image_paths)

# --- Creazione Dataset e DataLoader ---
def collate_fn(batch):
    return tuple(zip(*batch))

# Istanziamo i dataset passando le liste di train e val generate dallo split
train_dataset = GlomeruliDataset(train_imgs, train_masks)
val_dataset = GlomeruliDataset(val_imgs, val_masks)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

### 2. Mask R-CNN Architecture Setup
We load a ResNet50-FPN backbone pre-trained on COCO. We must modify the two output heads (Box Predictor and Mask Predictor) because the original model predicts 91 classes, while we only need 2 (Background, Glomerulus).

In [ ]:
def get_model_instance_segmentation(num_classes):
    # Load an instance segmentation model pre-trained on COCO
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

    # --- Replace Bounding Box Predictor ---
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # --- Replace Mask Predictor ---
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model

# 2 classes (Background = 0, Glomerulus = 1)
num_classes = 2
model = get_model_instance_segmentation(num_classes)
model.to(device)
print("Model built and moved to device.")

### 3. Model Training
In PyTorch, when Mask R-CNN is in `.train()` mode and receives both images and targets, it automatically computes a dictionary of 4 losses:
1. `loss_classifier`: accuracy of the class prediction.
2. `loss_box_reg`: accuracy of the bounding box coordinates.
3. `loss_mask`: accuracy of the pixel-wise mask.
4. `loss_objectness`: RPN's ability to find regions of interest.

In [ ]:
from tqdm import tqdm # Aggiungi questo in alto nel tuo script

# Setup Optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

num_epochs = 15 

print("Starting Training...")

# Liste per i grafici
train_loss_history = []
val_loss_history = []
detailed_losses = {'loss_classifier': [], 'loss_box_reg': [], 'loss_mask': [], 'loss_objectness': []}

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    
    # --- FASE 1: TRAINING ---
    model.train() 
    epoch_train_loss = 0
    epoch_details = {k: 0.0 for k in detailed_losses.keys()}
    
    # Barra di progresso per il Training
    train_loop = tqdm(train_loader, desc="Training  ", leave=False)
    
    for images, targets in train_loop:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()    
        optimizer.step()     
        
        epoch_train_loss += losses.item()
        for k in loss_dict.keys():
            if k in epoch_details:
                epoch_details[k] += loss_dict[k].item()
                
        # Aggiorna la loss mostrata nella barra di progresso
        train_loop.set_postfix(loss=f"{losses.item():.4f}")
        
    avg_train_loss = epoch_train_loss / len(train_loader)
    train_loss_history.append(avg_train_loss)
    
    for k in detailed_losses.keys():
        detailed_losses[k].append(epoch_details[k] / len(train_loader))
        
    
    # --- FASE 2: VALIDATION ---
    epoch_val_loss = 0
    
    # Barra di progresso per la Validation
    val_loop = tqdm(val_loader, desc="Validation", leave=False)
    
    with torch.no_grad():
        for val_images, val_targets in val_loop:
            val_images = list(image.to(device) for image in val_images)
            val_targets = [{k: v.to(device) for k, v in t.items()} for t in val_targets]
            
            val_loss_dict = model(val_images, val_targets)
            val_losses = sum(loss for loss in val_loss_dict.values())
            
            epoch_val_loss += val_losses.item()
            
            # Aggiorna la loss mostrata nella barra di progresso
            val_loop.set_postfix(val_loss=f"{val_losses.item():.4f}")
            
    avg_val_loss = epoch_val_loss / len(val_loader)
    val_loss_history.append(avg_val_loss)
    
    # --- STAMPA RISULTATI FINALI EPOCA ---
    print(f"--> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Mask Loss (Train): {detailed_losses['loss_mask'][-1]:.4f}")

print("\nTraining Complete.")

# --- PLOT LOSS: Train vs Validation ---
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), train_loss_history, marker='o', label='Training Loss', color='blue')
plt.plot(range(1, num_epochs+1), val_loss_history, marker='s', label='Validation Loss', color='red', linestyle='--')

plt.title("Mask R-CNN: Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Total Loss")
plt.legend()
plt.grid(True)
plt.show()

# Save the trained model
torch.save(model.state_dict(), 'mask_rcnn_glomeruli.pth')
print("Model saved to 'mask_rcnn_glomeruli.pth'")

### 4. Inference and Visualization
We switch the model to `.eval()` mode. In this mode, providing just an image returns predictions (boxes, scores, and masks). We apply a confidence threshold to filter out uncertain detections and visualize the results.

In [ ]:
model.eval()

# Grab one image from the validation set
img, target = val_dataset[0]
img_tensor = img.unsqueeze(0).to(device)

with torch.no_grad():
    prediction = model(img_tensor)[0] # Get predictions for the first (and only) image in batch

# Move image to numpy for matplotlib (H, W, C)
img_np = img.permute(1, 2, 0).numpy()

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# Plot Original Image
ax[0].imshow(img_np)
ax[0].set_title("Original Input Image")
ax[0].axis("off")

# Plot Image with Predictions
ax[1].imshow(img_np)
ax[1].set_title("Identification & Segmentation (Mask R-CNN)")
ax[1].axis("off")

confidence_threshold = 0.6 # Only show predictions with > 60% confidence

for i in range(len(prediction['scores'])):
    score = prediction['scores'][i].cpu().numpy()
    if score >= confidence_threshold:
        
        # 1. SEGMENTATION: Draw Mask
        mask = prediction['masks'][i, 0].cpu().numpy()
        mask_binary = mask > 0.5
        
        # Create a green overlay for the mask
        overlay = np.zeros_like(img_np)
        overlay[mask_binary] = [0, 1, 0] # Green
        ax[1].imshow(overlay, alpha=0.4)
        
        # 2. IDENTIFICATION: Draw Bounding Box
        box = prediction['boxes'][i].cpu().numpy()
        xmin, ymin, xmax, ymax = box
        rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, 
                                 linewidth=2, edgecolor='red', facecolor='none')
        ax[1].add_patch(rect)
        ax[1].text(xmin, ymin - 5, f"Glomerulus: {score:.2f}", color='red', fontsize=12, weight='bold')

plt.tight_layout()
plt.show()

In [ ]:
def calculate_iou(pred_mask, true_mask):
    """Calcola l'Intersection over Union tra due maschere binarie"""
    intersection = np.logical_and(pred_mask, true_mask).sum()
    union = np.logical_or(pred_mask, true_mask).sum()
    if union == 0:
        return 0.0
    return intersection / union

# Esempio di utilizzo (quando fai inferenza):
true_mask = target['masks'][0].cpu().numpy()
pred_mask = mask_binary
iou = calculate_iou(pred_mask, true_mask)
print(f"L'IoU di questo glomerulo è: {iou:.2f}")

In [ ]:
import openslide
import torch
import torchvision
import numpy as np
from torchvision.transforms import functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 1. IMPOSTAZIONI
WSI_PATH = "./glomeruli_grading/RECHERCHE-003.svs"
PATCH_SIZE = 512
STRIDE = 256  # Spostamento della finestra (overlap del 50%)
CONFIDENCE_THRESHOLD = 0.7

# (Assumo che model e device siano già inizializzati e il modello sia in model.eval())

slide = openslide.OpenSlide(WSI_PATH)
wsi_width, wsi_height = slide.dimensions

global_boxes = []
global_scores = []
# global_masks = [] # NOTA: Salvare le maschere di tutta la WSI satura la RAM. Di solito si salvano solo i box, o le maschere compresse.

print(f"Inizio scansione WSI {wsi_width}x{wsi_height}...")

# 2. SLIDING WINDOW ITERATION
with torch.no_grad():
    for y in range(0, wsi_height, STRIDE):
        for x in range(0, wsi_width, STRIDE):
            
            # Se stiamo uscendo dal bordo destro/basso, aggiustiamo
            if x + PATCH_SIZE > wsi_width or y + PATCH_SIZE > wsi_height:
                continue 
            
            # Leggiamo il patch
            patch = slide.read_region((x, y), 0, (PATCH_SIZE, PATCH_SIZE)).convert("RGB")
            img_tensor = F.to_tensor(patch).unsqueeze(0).to(device)
            
            # Inferenza della Mask R-CNN
            prediction = model(img_tensor)[0]
            
            # 3. CONVERSIONE DA COORDINATE LOCALI A GLOBALI
            for i in range(len(prediction['scores'])):
                score = prediction['scores'][i].item()
                if score >= CONFIDENCE_THRESHOLD:
                    box = prediction['boxes'][i].cpu().numpy()
                    
                    # Il box predetto è relativo al patch 512x512. 
                    # Lo trasliamo aggiungendo le coordinate (x, y) di dove ci troviamo nella WSI!
                    global_xmin = box[0] + x
                    global_ymin = box[1] + y
                    global_xmax = box[2] + x
                    global_ymax = box[3] + y
                    
                    global_boxes.append([global_xmin, global_ymin, global_xmax, global_ymax])
                    global_scores.append(score)

print(f"Trovati {len(global_boxes)} glomeruli (con possibili duplicati da overlap).")

# 4. NON-MAXIMUM SUPPRESSION (NMS) PER RIMUOVERE I DUPLICATI
boxes_tensor = torch.tensor(global_boxes, dtype=torch.float32)
scores_tensor = torch.tensor(global_scores, dtype=torch.float32)

# La NMS tiene un solo box se l'IoU tra due box è superiore a 0.3
keep_indices = torchvision.ops.nms(boxes_tensor, scores_tensor, iou_threshold=0.3)

final_boxes = boxes_tensor[keep_indices].numpy()
final_scores = scores_tensor[keep_indices].numpy()

print(f"Dopo la NMS, sono rimasti {len(final_boxes)} glomeruli unici.")

# 5. VISUALIZZAZIONE SULLA WSI A BASSA RISOLUZIONE
# Dato che la WSI è enorme, per plottarla la rimpiccioliamo leggendo un livello più basso (es. livello 2 o 3)
plot_level = slide.level_count - 1 # Prende il livello di zoom più basso
downsample_factor = slide.level_downsamples[plot_level]
wsi_thumbnail = slide.read_region((0,0), plot_level, slide.level_dimensions[plot_level]).convert("RGB")

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.imshow(wsi_thumbnail)

# Disegniamo i box, ma ricordiamoci di rimpicciolire anche le loro coordinate!
for box, score in zip(final_boxes, final_scores):
    xmin, ymin, xmax, ymax = box / downsample_factor
    rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, 
                             linewidth=1.5, edgecolor='blue', facecolor='none')
    ax.add_patch(rect)

ax.set_title(f"WSI Inference - Trovati {len(final_boxes)} Glomeruli")
ax.axis("off")
plt.show()

slide.close()